# Training (fine-tuning) the model 

Finally, we're at the step you've been waiting for.  Let's put it all together ... 

## Imports 

Import the necessarily libraries.  

In [ ]:
import yaml
import os

import matplotlib.pyplot as plt
import pandas as pd
import torch

from typing import Any
from pprint import pprint
from textwrap import dedent
from datasets import load_dataset
from peft import LoraConfig
from transformers import (
    AutoModelForCausalLM,
    TrainingArguments,
    EarlyStoppingCallback
)
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from trl import DataCollatorForCompletionOnlyLM, SFTTrainer

# To make prettier plots!
plt.style.use("ggplot")

## Configuration 

Detect your device (GPU/CPU) and define the model to use - you've seen all this before 😊

In [ ]:
def get_device():
    """Determine the available device for computation."""
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    return "cpu"

MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"
DEVICE = get_device()

Load configuration options from a YAML file.

In [ ]:
# Load configuration parameters from a YAML file into a dictionary.
# This file typically contains parameters such as learning rates, batch sizes, model names, etc., used for fine-tuning.
with open("./finetuning_params.yml", "r") as file:
    config: dict[str, Any] = yaml.safe_load(file)

print("Config loaded")

# Retrieve model configuration parameters from `config`.
# Update the `device_map` to ensure the model loads on the selected device.
model_kwargs = config["model_kwargs"]
model_kwargs["device_map"] = DEVICE

# Display the model configuration parameters for verification and debugging purposes.
print(model_kwargs)

This should be familiar too 😊

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)

print(model.device)

## Load the tokenizer 

We described this previously in [tokenizer configuration](./07-model-and-tokenizer-config.ipynb)

In [ ]:
from transformers import AutoTokenizer

def load_tokenizer(model_name, model_kwargs=None):
    """Load and configure a tokenizer for function calling tasks."""
    model_kwargs = model_kwargs or {}  # Ensure kwargs is a dictionary if None

    tokenizer = AutoTokenizer.from_pretrained(model_name, **model_kwargs)

    # Set the maximum number of output tokens for the tokenizer to prevent unnecessary
    # memory usage. 2048 tokens are sufficient for function calling scenarios.
    tokenizer.model_max_length = 2048

    # Configure padding token (use unk_token to prevent endless generation)
    tokenizer.pad_token = tokenizer.unk_token or tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)

    # Use right-side padding for better batch processing
    tokenizer.padding_side = "right"

    return tokenizer

tokenizer = load_tokenizer(MODEL_NAME)

## Load the finetuning dataset

We're loading the cleaned dataset from [preparing your data](./05-preparing-our-data.ipynb) 😅

In [ ]:
# Load the training dataset from a CSV file.
# The "split='all'" option indicates that the entire dataset will be loaded without splitting it into train/test/validation.
train_dataset = load_dataset("csv", data_files="../Data/clean_train_data.csv", split="all")

# Load the test dataset from a separate CSV file.
# This dataset will be used to evaluate the model's performance.
test_dataset = load_dataset("csv", data_files="../Data/clean_test_data.csv", split="all")

print('Train and test datasets loaded')


In [ ]:
# Take a look at the training dataset, to refresh your memory:
train_dataset


### Convert the dataset to the format that the model need for the finetuning task

Apply a chat template as described in [chat templates](./06-chat-templates.ipynb)

In [ ]:
# Define a system prompt for an in-car virtual assistant.
# This prompt guides the behavior of the assistant by specifying how it should respond to user inputs.
SYSTEM_PROMPT = dedent(
    """
    You are an in car virtual assistant that maps user's inputs to the corresponding function call in the vehicle. You must respond with only a JSON object matching the following schema:

    {
        "function_name": <name of the function>,
        "parameters": {
            <arg1_name>: <arg1_value>,
            <arg2_name>: <arg2_value>,
            ...
        }
    }

    Important: DO NOT include any other information in the response.
    """
)

In [ ]:
def apply_chat_template(
    example: dict[str, Any],
    tokenizer: PreTrainedTokenizerBase,
) -> dict[str, Any]:
    """
    Prepares an input example for a chat-based model by applying a chat template.

    Args:
    -----
    example (dict[str, Any]): A dictionary containing the input example. It must include keys like:
        - "prompt": The user's input or question.
        - "ground_truth": The expected response or reference answer from the assistant.
    tokenizer (PreTrainedTokenizerBase): A tokenizer object from the Hugging Face Transformers library.
        It is used to format and prepare the input for the model.

    Returns:
    --------
    dict[str, Any]: The updated example dictionary with a new "text" key that contains the formatted
                    input for the chat-based model.
    """
    # Create a message chain that defines the context and dialogue flow for the chat-based model.
    # This includes:
    # 1. A system prompt to instruct the assistant on how to behave or respond.
    # 2. A user message from the example's "prompt" key.
    # 3. An assistant message representing the ground truth response.
    message_chain = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,  # System-level instruction for the assistant's behavior.
        },
        {
            "role": "user",
            "content": example["prompt"],  # User-provided input or question.
        },
        {
            "role": "assistant",
            "content": example[
                "ground_truth"
            ],  # Reference response from the assistant.
        },
    ]

    # Apply the chat template using the tokenizer.
    # - `message_chain`: The structured conversation to be tokenized.
    # - `tokenize=False`: Keeps the text in its raw form instead of tokenizing it.
    # - `add_generation_prompt=False`: Prevents adding a generation-specific prompt, leaving the response open-ended.
    example["text"] = tokenizer.apply_chat_template(
        message_chain,
        tokenize=False,
        add_generation_prompt=False,
    )

    # Return the updated example dictionary with the "text" key added.
    return example

In [ ]:
# Retrieve the column names from the train dataset to determine which columns to remove during processing
column_names = list(train_dataset.features)

# Process the training dataset by applying the chat template to each sample
# - `apply_chat_template`: A function that formats each dataset entry for a chat-based model.
# - `fn_kwargs={"tokenizer": tokenizer}`: Passes the tokenizer as an argument to the `apply_chat_template` function.
# - `num_proc=10`: Parallelizes the processing across 10 cores for faster execution.
# - `remove_columns=column_names`: Removes the original columns from the dataset after processing to save memory.
# - `desc="Applying chat template to train"`: Provides a description for progress logging.
processed_train_dataset = train_dataset.map(
    apply_chat_template,
    fn_kwargs={"tokenizer": tokenizer},
    num_proc=10,
    remove_columns=column_names,
    desc="Applying chat template to train",
)

# Process the test dataset in the same way as the training dataset
# This ensures consistency in how both datasets are prepared for use with the chat-based model.
processed_test_dataset = test_dataset.map(
    apply_chat_template,
    fn_kwargs={"tokenizer": tokenizer},
    num_proc=10,
    remove_columns=column_names,
    desc="Applying chat template to test",
)


## Train  🚀

This section configures the training parameters, initializes the model training process, and logs critical details about the training configuration. It also includes provisions for training on completions only using a custom data collator. The model is fine-tuned using the Hugging Face `SFTTrainer` with PEFT (Parameter-Efficient Fine-Tuning).


### Explore the training configuration

#### Training strategy

When data scientists refer to a “training strategy”, they usually refer to the combination of choices and techniques used to train a model efficiently, effectively, and within hardware constraints.

Training an AI model isn’t just about running code—it involves planning and decision-making to balance:

1. Compute efficiency (making the most of your CPU/GPU resources)
2. Model performance (getting the best accuracy or loss improvement)
3. Memory constraints (avoiding crashes due to out-of-memory errors)
4. Training time (shortening training while maintaining quality)

Let's take a look at the configuration: 

In [ ]:
# Retrieve and display the training configuration from the config file
training_config = config["training_config"]
pprint(training_config)

Wow, that escalated quickly 😂   

Key parameters to note:

- `learning_rate`: The learning rate is set to 2e-5, which is a common starting point for fine-tuning transformer models. It controls how much to adjust the model weights with respect to the loss gradient. Too big a value can cause the model to converge too quickly to a suboptimal solution, while too small a value can make the training process unnecessarily slow.

- `num_train_epochs`: The number of epochs is set to 5. An epoch is one complete pass through the training dataset. Depending on the size of the dataset and the model, this can be adjusted up or down. Too much training can lead to unecessary training runs (sometimes can led to memorisation without generalisation), while too little can result in the model not learning enough.

**Optional**: Here is a brief description of all the parameters, categorized by area of impact. (Don't worry, there **won't** be a test on this later - feel free to trust us and skip the gory details!)

<details>
    <summary>Click to expand</summary>

1. General Training Setup
	- bf16: True → Uses bfloat16 (BF16) precision for faster training and lower memory usage.
	- num_train_epochs: 5 → The model will be trained for 5 full passes over the dataset.
	- max_steps: -1 → This means training will run for full epochs, instead of stopping after a fixed number of steps.

2. Batch Sizes & Gradient Accumulation
	- per_device_train_batch_size: 8 → Each GPU processes 8 samples per batch during training.
	- per_device_eval_batch_size: 8 → Each GPU processes 8 samples per batch during evaluation.
	- gradient_accumulation_steps: 4 → Gradients are accumulated over 4 steps before updating model weights.
	- This makes training more memory-efficient, as effective batch size =  8 \times 4 = 32 .

3. Optimizer & Learning Rate
	- learning_rate: 2.5e-05 (0.000025) → Initial learning rate for fine-tuning.
	- lr_scheduler_type: ‘cosine’ → Learning rate follows a cosine decay schedule, meaning it starts high and gradually decreases.
	- warmup_ratio: 0.2 → 20% of training is used for warm-up, where the learning rate gradually increases before reaching its peak.

4. Checkpointing & Logging
	- output_dir: ‘./models/checkpoint_dir’ → Checkpoints and model outputs are saved here.
	- overwrite_output_dir: True → Overwrites previous checkpoints if they exist.
	- save_steps: 10 → Saves a checkpoint every 10 steps.
	- logging_steps: 1 → Logs metrics (loss, accuracy, etc.) every step.
	- log_level: ‘warning’ → Logs only warnings and errors (no verbose logs).
	- logging_strategy: ‘steps’ → Logs at fixed steps rather than by epoch.
	- report_to: ‘none’ → Disables reporting to external tracking tools like Weights & Biases.

5. Evaluation & Best Model Selection
	- do_eval: True → Runs evaluation during training.
	- eval_strategy: ‘steps’ → Evaluates after a fixed number of steps (not after each epoch).
	- eval_steps: 1 → Evaluates every single training step (very frequent).
	- load_best_model_at_end: True → After training, restores the best-performing checkpoint based on evaluation metrics.

</details>

<span style="color: green;">**QUESTION**</span>: what's the difference between a "step" and an "epoch" ? 

<details>
    <summary>Click for the answer</summary>

#### What is a “Step” in training?

A step refers to one update to the model’s weights. It occurs every time the model processes a batch of data.

💡 1 step = 1 forward pass + 1 backward pass (gradient update) over a batch of data.

So, if you have:
- Batch size = 8
- 10,000 training samples

Then:
- 1 step processes 8 samples
- 1,250 steps process all 10,000 samples (one full epoch)

📌 Key point: A step happens every time the model processes a batch.


#### What is an “Epoch”?

An epoch means one full pass through the entire dataset.

So, if you have:
- Batch size = 8
- 10,000 training samples 

Then: 
- It takes 1,250 steps to complete 1 epoch
- For 5 epochs, the model will process 50,000 samples in total (with repetition).

📌 Key point: The number of epochs defines how many times the model sees the full dataset.

</details>

In [ ]:
# Retrieve and display the PEFT configuration for parameter-efficient fine-tuning
peft_config = config["peft_config"]
pprint(peft_config)

Some important parameters to note:
- `lora_alpha`: The LoRA alpha is set to 32. This is a hyperparameter that controls the scaling of the low-rank matrices in the LoRA method. A higher value can lead to more significant updates to the model weights, while a lower value can make the updates more conservative.
- `lora_r`: The LoRA rank is set to 16. This determines the rank of the low-rank matrices used in the LoRA method. A higher rank allows for more complex updates to the model weights, while a lower rank can lead to faster training but potentially less expressive power.

In [ ]:
# Train for 3 steps for illustration purpose
training_config["max_steps"] = 3

# If you want to run the full training, uncomment the line below
# training_config["max_steps"] = -1

In [ ]:
# Initialize training arguments with the parameters from the configuration
train_conf = TrainingArguments(**training_config)
# Initialize PEFT configuration for LoRA (Low-Rank Adaptation)
peft_conf = LoraConfig(**peft_config)

# Print a summary of the training setup for each process (useful for distributed training)
print(
    ", ".join(
        [
            f"Process rank: {train_conf.local_rank}",  # Rank of the process in distributed training
            f"device: {train_conf.device}",  # Device being used (e.g., CUDA or CPU)
            f"n_gpu: {train_conf.n_gpu}",  # Number of GPUs available
            f"distributed training: {bool(train_conf.local_rank != -1)}",  # Whether distributed training is enabled
        ]
    )
)

In [ ]:
# Check if training is restricted to completion-only data
train_on_completion_only = config.get("train_on_completion_only", False)
if train_on_completion_only:
    # Retrieve the response template required for completion-only training
    response_template = config.get("response_template", None)
    if response_template is None:
        raise ValueError(
            "response_template is required in the config file when train_on_completion_only is True."
        )
    print(f"Using response template: {response_template}")

    # Initialize a custom data collator for completion-only fine-tuning
    collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)
    print("Using DataCollatorForCompletionOnlyLM")
else:
    # Use the default data collator if not restricted to completion-only training
    collator = None
    print("Using default data collator")

### Create the Trainer 

Initialize and configure the SFTTrainer for fine-tuning a model using Parameter-Efficient Fine-Tuning (PEFT).

You've met most of these parameters above. 

In [ ]:
# Ensure the output directory exists to store training outputs
output_dir = training_config["output_dir"]
os.makedirs(output_dir, exist_ok=True)

# Log that the trainer is being created
print("Creating trainer")

# Initialize the `SFTTrainer` for fine-tuning
trainer = SFTTrainer(
    model=model,  # Pre-trained model
    args=train_conf,  # Training arguments
    peft_config=peft_conf,  # PEFT configuration for fine-tuning
    train_dataset=processed_train_dataset,  # Training dataset
    eval_dataset=processed_test_dataset,  # Evaluation dataset
    max_seq_length=2048,  # Maximum sequence length for inputs
    dataset_text_field="text",  # Field in the dataset containing the text
    processing_class=tokenizer,  # Tokenizer for pre-processing
    data_collator=collator,  # Data collator (custom or default)
    packing=False,  # Disable packing as per Hugging Face TRL guidelines
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=config.get("early_stopping_patience", 20),
            early_stopping_threshold=0.001,
        ),
    ],
)

### Start the training process 🚀

Here you go, the moment you've been waiting for! 🤗 

In [ ]:
if torch.cuda.is_available():
    total_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if total_memory >= 16:
        print(f"CUDA device has {total_memory:.2f} GB of memory available. You're good to go!")
    else:
        print(f"WARNING: CUDA device has only {total_memory:.2f} GB of memory. At least 16 GB is recommended.")

elif torch.backends.mps.is_available() and training_config["max_steps"] <= 3:
    print("Looks like you've got Apple Silicon with metal peformance shaders. You should be able to finetune 3 steps. But still, expect this to be slow.")

else:
    warning_msgs = [
        "WARNING: CUDA GPU IS NOT AVAILABLE, TRAINING WILL ALMOST CERTAINLY NOT COMPLETE EVEN OVERNIGHT, OR YOUR NOTEBOOK MAY CRASH.",
        "IF YOU STILL WANT TO PROCEED, IGNORE THIS WARNING AND RUN THE CODE BELOW."
    ]

    from IPython.display import Image, display
    display(Image(filename="images/stop.png", width=600))

    for warning_msg in warning_msgs:
        print(f"\033[1m\033[91m{warning_msg}\033[0m")

    raise Exception("Force stopping the code execution to make sure you get the message :)")


In [ ]:
print("Training adapter")
train_result = trainer.train()  # Train the model

### Save the adapted model 

`trainer.save_model(...)` saves the fine-tuned LoRA adapter weights. Note that it does NOT save the full model (like in full fine-tuning). Instead, it saves:

- LoRA adapter weights (small, efficient fine-tuning layers).
- PEFT config (so it can be reloaded later).
- Training arguments (metadata about training settings).

This has the advantage that LoRA adapters are tiny compared to full model weights (e.g., MBs instead of GBs), faster loading and memory efficient. 

In [ ]:
# Save the trained adapter
print("Saving adapter")
trainer.save_model(config["final_model_dir"])

--------

## Analyse training results

### If you have run finetuning

<span style="color: red;">**WARNING**</span>: Only run the cells below in this section if you have successfully run the training process for at least 3 steps.

Otherwise, skip to this section: [Analysing the loss curve](#analysing-the-loss-curves)

In [ ]:
print("Final training result metrics")
print(train_result.metrics)

In [ ]:
print("Result of the first 3 training iterations")
trainer.state.log_history[:3]

The data format is not really easy to see any trend, or very readable. Let's massage it into a tabular format, and plot it

In [ ]:
train_history = {}
metrics = {}
step = 0

for entry in trainer.state.log_history[
    :-1
]:  # Skip the last entry, because it's the final result which we've already extracted
    if entry["step"] not in train_history:
        train_history[entry["step"]] = {}

    train_history[entry["step"]].update(entry)

train_history_df = pd.DataFrame(train_history).T

train_history_df.head(5)

In [ ]:
train_history_df.plot(
    x="step",
    y=["loss", "eval_loss"],
    title="Training and Evaluation Loss Over Time",
)
plt.show()

### Analysing the loss curves

If you have successfully run the **full** finetuning process, the above code cell will produce a plot that looks similar to this one:

![](images/fully_trained.png)

This plot shows the training loss (red curve) and the evaluation loss (blue curve) over the course of fine-tuning our model.

<span style="color: green;">**QUESTION**</span>: Do you know what these mean? If you don't, does the person next to you know?

<details>
    <summary>Click to show the answer</summary>
    
**`loss` (Training Loss)**

The training loss represents how well the model is performing on the training dataset. It is computed by comparing the model's predictions against the ground truth labels using a loss function (e.g., cross-entropy loss for language models). The goal of training is to minimize this loss over time by updating the model’s parameters using optimization algorithms like gradient descent.
 

**`eval_loss` (Evaluation Loss or Validation Loss)**

The evaluation loss represents how well the model is performing on the test dataset (not used for training).

</details>

<span style="color: green;">**QUESTION**</span>: What does the plot tell us about the model's training progress? How did it go?

<details>
    <summary>Click to show the answer</summary>

Based on the plot:

**Both losses are decreasing**

This indicates that the model is learning well and improving its performance both on the training data and on the validation data.
The fact that the evaluation loss closely tracks the training loss suggests that the model is not overfitting (a common issue where the model performs well on training data but poorly on validation data). However, in this case, the two curves are very close, indicating good generalization.
 
**Loss convergence**

Both curves steadily decrease and start to flatten out near zero as the number of steps increases. This flattening suggests that the model is approaching its optimal performance and no further improvement can be made.

Therefore, we can say that the training process is going well:

The loss is steadily decreasing, suggesting that the model is learning effectively.
The close alignment of training loss and eval loss indicates that the model is generalizing well to unseen data.
If this trend continues, you can expect the model to perform well on downstream tasks.
</details>


<span style="color: green;">**QUESTION**</span>: And what about this one? What's happening here?

![](images/not_full_train.png)

<details>
    <summary>Click to show the answer</summary>

This plot shows a different behavior in the training process compared to the previous one. Let’s analyze what’s happening with the training loss (red curve) and the evaluation loss (blue curve):
 
**The training loss (red curve):**

Decreases quickly at the start, which is expected as the model adjusts its weights to minimize the loss.
After a steep drop, the training loss exhibits sharp oscillations, with occasional spikes. This suggests some instability in the optimization process (e.g., large weight updates or irregular gradients).
Despite the oscillations, the overall trend is downward, and the training loss approaches near-zero values by the end.

**The evaluation loss (blue curve):**

Decreases consistently early on, but flattens out after a certain point.
The evaluation loss remains much higher than the training loss, which suggests that the model is overfitting—i.e., it is performing well on the training data but not generalizing as effectively to the validation data.
 
**Overfitting**

The training loss is almost zero by the end of training, but the evaluation loss remains significantly higher. This is a clear sign of overfitting, where the model has learned to "memorize" the training data instead of generalizing to unseen data.

Overfitting often happens when:

- The training data is too small or not representative of the validation set.
- The model is too complex (e.g., too many parameters) relative to the size of the dataset.
- Regularization techniques (e.g., dropout, weight decay) are not being used effectively.

In this case, it was because the dataset was too small. I only trained on the first 100 examples out of 4000 in the training dataset.
</details>

<span style="color: green;">**BONUS QUESTION**</span>: How do you mitigate this issue? (hint: ask someone next to you, or a data scientist)

<details>
    <summary>Click to show the answer</summary>

Use regularization techniques:
  - Add dropout layers to the model to prevent over-reliance on specific neurons.
  - Apply weight decay (L2 regularization) to constrain the size of the model weights.
- Collect more training data or augment the existing dataset to make it more representative.
- Consider early stopping based on validation loss to halt training before overfitting occurs.
</details>
